### Cell 01 - Import Dependencies and Basic Settings

This cell imports libraries for numerical computing, plotting, table I/O, and GP-HT-related routines, and sets basic paths or plotting styles used later.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.
- Main parameters: INPUT_EXCEL, OUTPUT_DIR, DATA_TYPES, PROCESSING_TYPES, PARAM_NAMES, DROP_IM_POSITIVE, MIN_POINTS, SORT_BY_FREQ_ASC, WEIGHT_MODE_MAIN, REL_SCALE_FLOOR, LOSS_MAIN, F_SCALE, N_RANDOM_STARTS, N_GROUP_REFIT_STARTS, RANDOM_SEED, RUN_BOOTSTRAP, N_BOOTSTRAP, BOOTSTRAP_MODE.These variables control input/output paths, experimental conditions, frequency ranges, or plotting behavior.


In [ ]:
# Cell 1. , Pathand
import os
import re
import math
import json
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy import stats
warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams["font.sans-serif"] = ["Arial", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False
# ===== Input/output paths and case-data files =====
NOTEBOOK_DIR = Path.cwd()
CASE_DATA_DIR = NOTEBOOK_DIR.parent / "6 Case Data" if (NOTEBOOK_DIR.parent / "6 Case Data").exists() else NOTEBOOK_DIR / "6 Case Data"  # This is the author's local input/output path; please update it before running.
DATASET_NAME = "mouse"  # Options: "cell", "potato", "mouse"。
CASE_DATA_FILES = {
    "cell": CASE_DATA_DIR / "case_data_cell.xlsx",
    "potato": CASE_DATA_DIR / "case_data_potato.xlsx",
    "mouse": CASE_DATA_DIR / "case_data_mouse.xlsx",
}
INPUT_EXCEL = CASE_DATA_FILES[DATASET_NAME]  # This is the author's local input/output path; please update it before running.
OUTPUT_DIR = Path(r"C:/Users/CYJ/Desktop/Experimental_Data_DoubleCole_outputs")  # This is the author's local input/output path; please update it before running.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_TYPES = ["raw", "sparse", "limited", "noisy"]
PROCESSING_TYPES = ["Exp", "GP"]
PARAM_NAMES = ["R_inf", "R1", "tau1", "alpha1", "R2", "tau2", "alpha2"]
# ===== data =====
DROP_IM_POSITIVE = True     # EIS in Im(Z) as; if of data -Im, as False
MIN_POINTS = 12  # Minimum number of valid frequency points required for analyzing one spectrum.
SORT_BY_FREQ_ASC = True
# ===== Fitting objective function =====
# modulus_relative: residual |Z|, to avoidlow-frequencyimpedance; used for
# component_relative: real component |Re|, imaginary component |Im|; for 0 of imaginary component
# unweighted: use direct differences; not recommended for the main analysis and intended only for sensitivity analysis
WEIGHT_MODE_MAIN = "modulus_relative"
REL_SCALE_FLOOR = 0.05      # scale = max(|Z|, floor*median(|Z|)), to avoidhigh-frequencyvalue
LOSS_MAIN = "soft_l1"
F_SCALE = 1.0
# ===== Multi-start settings =====
N_RANDOM_STARTS = 20
N_GROUP_REFIT_STARTS = 60
RANDOM_SEED = 20260616
rng = np.random.default_rng(RANDOM_SEED)
# ===== Bootstrap/profile/sensitivity settings =====
RUN_BOOTSTRAP = True
N_BOOTSTRAP = 200           # Use 50-100 first for debugging.
BOOTSTRAP_MODE = "residual_pair"  # residual_pair: resample paired complex residuals
RUN_PROFILE_LIKELIHOOD = True     # Time-consuming; enable for selected key samples in formal runs.
PROFILE_GRID_N = 25
RUN_SENSITIVITY = True
SENSITIVITY_WEIGHT_MODES = ["modulus_relative", "component_relative", "unweighted"]
# ===== Jump/outlier checks =====
# parameterrelative group median when refit.tau log10, R fold change.
ENABLE_GROUP_GUIDED_REFIT = True
RESISTANCE_FOLD_THRESHOLD = 5.0
TAU_LOG10_THRESHOLD = 1.5
ALPHA_ABS_THRESHOLD = 0.25
ACCEPT_REFIT_COST_INCREASE = 1.05  # Accept the group-guided refit only if its cost is no more than 5% higher than the original cost.
# ===== outputdirectory =====
DATA_DIR = OUTPUT_DIR / "tables"  # This is the author's local input/output path; please update it before running.
FIG_DIR = OUTPUT_DIR / "figures"  # This is the author's local input/output path; please update it before running.
CURVE_DIR = DATA_DIR / "fit_curves"  # This is the author's local input/output path; please update it before running.
for d in [DATA_DIR, FIG_DIR, CURVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)


### Cell 02 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.

- Function descriptions:
  - `_safe_array(x)`: Converts the input object into a finite NumPy array used by later fitting and quality-control steps.
  - `clean_impedance(freq, z, drop_im_positive, min_points)`: Removes invalid impedance values, sorts the spectrum by frequency, and optionally filters positive imaginary points.
  - `load_gpht_excel(input_excel)`: Reads the GP-HT summary workbook and reorganizes it by sample, input type, and processing branch.

In [ ]:
# Cell 2. Read Excel And As Sample/Type/Processing
def _safe_array(x):
    return np.asarray(x, dtype=float)
def clean_impedance(freq, z, drop_im_positive=True, min_points=MIN_POINTS):
    """impedance: NaN/Inf, frequencysort, default Im(Z)>0 of point."""
    freq = _safe_array(freq)
    z = np.asarray(z, dtype=complex)
    mask = np.isfinite(freq) & np.isfinite(z.real) & np.isfinite(z.imag) & (freq > 0)
    if drop_im_positive:
        mask &= (z.imag <= 0)
    freq = freq[mask]
    z = z[mask]
    if SORT_BY_FREQ_ASC and len(freq) > 0:
        order = np.argsort(freq)
        freq = freq[order]
        z = z[order]
    if len(freq) < min_points:
        return np.array([]), np.array([], dtype=complex)
    return freq, z
def load_gpht_excel(input_excel: Path) -> Dict[str, Dict[str, Dict[str, dict]]]:
    """
    Read the existing GP-HT summary workbook.
    Default column-name format:
        {dtype}_Freq_Exp, {dtype}_Re_Exp, {dtype}_Im_Exp
        {dtype}_Freq_GP, {dtype}_Re_GP, {dtype}_Im_GP
    : data_store[sample][dtype][processing] = {'f': freq, 'z': complex impedance}
    """
    xls = pd.ExcelFile(input_excel)
    data_store = {}
    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet)
        data_store[sheet] = {}
        for dtype in DATA_TYPES:
            data_store[sheet][dtype] = {}
            for src in PROCESSING_TYPES:
                f_col = f"{dtype}_Freq_{src}"
                re_col = f"{dtype}_Re_{src}"
                im_col = f"{dtype}_Im_{src}"
                if not {f_col, re_col, im_col}.issubset(df.columns):
                    continue
                f = df[f_col].dropna().to_numpy(dtype=float)
                re_v = df[re_col].dropna().to_numpy(dtype=float)
                im_v = df[im_col].dropna().to_numpy(dtype=float)
                n = min(len(f), len(re_v), len(im_v))
                if n == 0:
                    continue
                f, z = clean_impedance(f[:n], re_v[:n] + 1j * im_v[:n], DROP_IM_POSITIVE)
                if len(f) > 0:
                    data_store[sheet][dtype][src] = {"f": f, "z": z}
    return data_store
data_store = load_gpht_excel(INPUT_EXCEL)
print(f"Loaded {len(data_store)} sheets/samples")
for s in list(data_store.keys())[:5]:
    print(s, {dt: list(data_store[s].get(dt, {}).keys()) for dt in DATA_TYPES})


### Cell 03 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.

- Function descriptions:
  - `double_cole_impedance(p, freq_hz)`: Takes frequency and double Cole-Cole parameters, and returns the double-relaxation complex impedance curve.
  - `sort_cole_params(p)`: Sorts the two Cole branches by time constant to prevent label switching between fitted branches.
  - `dynamic_bounds(freq, z)`: Builds parameter bounds automatically from the frequency range and impedance scale of the current spectrum.
  - `residual_vector(p, freq, z_target, weight_mode)`: Computes the real and imaginary residual vector for least-squares fitting under the selected weighting mode.
  - `fit_metrics(freq, z_data, p)`: Compares the fitted curve with the target impedance spectrum and returns weighted errors, complex relative error, and Re/Im NRMSE.

In [ ]:
# Cell 3. Cole-Cole, Residualandparametersort
def double_cole_impedance(p, freq_hz):
    """Double Cole-Cole / two-ZARC impedance model.freq_hz input as Hz."""
    R_inf, R1, tau1, alpha1, R2, tau2, alpha2 = np.asarray(p, dtype=float)
    w = 2 * np.pi * np.asarray(freq_hz, dtype=float)
    term1 = R1 / (1.0 + (1j * w * tau1) ** alpha1)
    term2 = R2 / (1.0 + (1j * w * tau2) ** alpha2)
    return R_inf + term1 + term2
def sort_cole_params(p):
    """ tau fromtosort Cole, to avoidparameter."""
    p = np.asarray(p, dtype=float).copy()
    R_inf, R1, tau1, alpha1, R2, tau2, alpha2 = p
    if tau1 <= tau2:
        return p
    return np.array([R_inf, R2, tau2, alpha2, R1, tau1, alpha1], dtype=float)
def dynamic_bounds(freq, z):
    """according tocurrentboundary."""
    freq = np.asarray(freq, dtype=float)
    z = np.asarray(z, dtype=complex)
    z_re = z.real
    re_min = max(0.0, np.nanpercentile(z_re, 1))
    re_max = max(np.nanpercentile(z_re, 99), re_min + 1e-6)
    width = max(re_max - re_min, np.nanmedian(np.abs(z)), 1.0)
    f_min, f_max = np.nanmin(freq), np.nanmax(freq)
    tau_min = 1.0 / (2 * np.pi * f_max) / 100.0
    tau_max = 1.0 / (2 * np.pi * f_min) * 100.0
    tau_min = max(tau_min, 1e-12)
    tau_max = min(max(tau_max, tau_min * 1000), 1e8)
    lower = np.array([0.0, 0.0, tau_min, 0.20, 0.0, tau_min, 0.20], dtype=float)
    upper = np.array([re_max + 3 * width, 5 * width, tau_max, 1.00, 5 * width, tau_max, 1.00], dtype=float)
    return lower, upper
def residual_vector(p, freq, z_target, weight_mode=WEIGHT_MODE_MAIN):
    """
    Return residuals used by least_squares.
    The main analysis uses modulus_relative by default:
        residual = (Z_model - Z_data) / max(|Z_data|, floor*median(|Z_data|))
    This prevents large low-frequency impedance points from dominating the fit and prevents extremely small high-frequency values from receiving unbounded weights.
    """
    p = sort_cole_params(p)
    z_model = double_cole_impedance(p, freq)
    diff = z_model - z_target
    if weight_mode == "unweighted":
        scale_re = scale_im = 1.0
        return np.r_[diff.real, diff.imag]
    if weight_mode == "component_relative":
        re_floor = REL_SCALE_FLOOR * np.nanmedian(np.abs(z_target.real) + 1e-30)
        im_floor = REL_SCALE_FLOOR * np.nanmedian(np.abs(z_target.imag) + 1e-30)
        scale_re = np.maximum(np.abs(z_target.real), re_floor)
        scale_im = np.maximum(np.abs(z_target.imag), im_floor)
        return np.r_[diff.real / scale_re, diff.imag / scale_im]
    mag_floor = REL_SCALE_FLOOR * np.nanmedian(np.abs(z_target) + 1e-30)
    scale = np.maximum(np.abs(z_target), mag_floor)
    return np.r_[diff.real / scale, diff.imag / scale]
def fit_metrics(freq, z_data, p):
    z_fit = double_cole_impedance(p, freq)
    mag_floor = REL_SCALE_FLOOR * np.nanmedian(np.abs(z_data) + 1e-30)
    scale = np.maximum(np.abs(z_data), mag_floor)
    rel_complex = np.sqrt(np.mean(np.abs((z_fit - z_data) / scale) ** 2))
    nrmse_re = np.sqrt(np.mean((z_fit.real - z_data.real) ** 2)) / (np.ptp(z_data.real) + 1e-30)
    nrmse_im = np.sqrt(np.mean((z_fit.imag - z_data.imag) ** 2)) / (np.ptp(z_data.imag) + 1e-30)
    mse_weighted = np.mean(residual_vector(p, freq, z_data, WEIGHT_MODE_MAIN) ** 2)
    return dict(weighted_mse=mse_weighted, rel_complex_rmse=rel_complex, nrmse_re=nrmse_re, nrmse_im=nrmse_im)


### Cell 04 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.

- Function descriptions:
  - `estimate_initial_tau_candidates(freq, z, n_candidates)`: Generates initial tau candidates from -Im peak positions and the available frequency range.
  - `make_start_points(freq, z, lower, upper, n_random, guided_center)`: Combines heuristic and random initial values to build multi-start fitting seeds.
  - `fit_double_cole_multistart(freq, z, weight_mode, n_random, guided_center)`: Fits one spectrum from multiple initial values and keeps the double Cole-Cole parameter set with the lowest loss.

In [ ]:
# Cell 4. Multi-Start Initial Valuegenerateandfunction
def estimate_initial_tau_candidates(freq, z, n_candidates=6):
    """according to -Im(Z) of and frequencyrangegenerate tau candidatevalue."""
    freq = np.asarray(freq, dtype=float)
    z = np.asarray(z, dtype=complex)
    neg_im = -z.imag
    candidates = []
    if len(freq) >= 5 and np.nanmax(neg_im) > 0:
        # Use the largest -Im points as candidate peaks to avoid relying on find_peaks
        idxs = np.argsort(neg_im)[-min(n_candidates, len(freq)):]
        for idx in idxs:
            if freq[idx] > 0:
                candidates.append(1.0 / (2 * np.pi * freq[idx]))
    # range log candidate
    f_min, f_max = np.nanmin(freq), np.nanmax(freq)
    tau_grid = np.logspace(np.log10(1/(2*np.pi*f_max)), np.log10(1/(2*np.pi*f_min)), n_candidates)
    candidates.extend(list(tau_grid))
    candidates = np.array([c for c in candidates if np.isfinite(c) and c > 0])
    return np.unique(np.clip(candidates, 1e-12, 1e8))
def make_start_points(freq, z, lower, upper, n_random=N_RANDOM_STARTS, guided_center=None):
    """generate multi-start initial value, containsinitial value, initial value and initial value."""
    z_re = z.real
    Rinf_est = max(lower[0] + 1e-12, np.nanpercentile(z_re, 5))
    R0_est = max(np.nanpercentile(z_re, 95), Rinf_est + 1.0)
    width = max(R0_est - Rinf_est, np.nanmedian(np.abs(z)), 1.0)
    tau_cands = estimate_initial_tau_candidates(freq, z)
    starts = []
    alpha_vals = [0.65, 0.80, 0.92]
    splits = [0.35, 0.50, 0.65]
    for t1 in tau_cands:
        for t2 in tau_cands:
            if t1 == t2:
                continue
            for split in splits[:2]:
                p0 = np.array([Rinf_est, width*split, t1, 0.80, width*(1-split), t2, 0.80])
                starts.append(p0)
            if len(starts) > 30:
                break
        if len(starts) > 30:
            break
    if guided_center is not None:
        gc = sort_cole_params(guided_center)
        starts.append(gc)
        # Add perturbed initial values around the group median
        for _ in range(max(10, n_random//4)):
            p = gc.copy()
            p[[0,1,4]] *= np.exp(rng.normal(0, 0.35, 3))
            p[[2,5]] *= 10 ** rng.normal(0, 0.4, 2)
            p[[3,6]] += rng.normal(0, 0.08, 2)
            starts.append(p)
    for _ in range(n_random):
        Rinf = rng.uniform(lower[0] + 1e-12, min(upper[0], max(upper[0]*0.6, lower[0]+1)))
        Rtot = rng.uniform(max(1.0, 0.2*width), max(2.0, 2.0*width))
        split = rng.uniform(0.2, 0.8)
        tau1 = 10 ** rng.uniform(np.log10(lower[2]), np.log10(upper[2]))
        tau2 = 10 ** rng.uniform(np.log10(lower[5]), np.log10(upper[5]))
        a1 = rng.uniform(0.55, 0.98)
        a2 = rng.uniform(0.55, 0.98)
        starts.append(np.array([Rinf, Rtot*split, tau1, a1, Rtot*(1-split), tau2, a2]))
    clipped = []
    for p in starts:
        p = sort_cole_params(np.asarray(p, dtype=float))
        p = np.minimum(np.maximum(p, lower + 1e-12), upper - 1e-12)
        clipped.append(p)
    return clipped
def fit_double_cole_multistart(freq, z, weight_mode=WEIGHT_MODE_MAIN, n_random=N_RANDOM_STARTS, guided_center=None):
    """point least_squares. best result andsuccessfulpoint."""
    freq, z = clean_impedance(freq, z, DROP_IM_POSITIVE)
    if len(freq) < MIN_POINTS:
        return None, pd.DataFrame()
    lower, upper = dynamic_bounds(freq, z)
    starts = make_start_points(freq, z, lower, upper, n_random=n_random, guided_center=guided_center)
    records = []
    best = None
    best_cost = np.inf
    for j, p0 in enumerate(starts):
        try:
            res = least_squares(
                residual_vector, p0, args=(freq, z, weight_mode),
                bounds=(lower, upper), loss=LOSS_MAIN, f_scale=F_SCALE,
                max_nfev=8000, xtol=1e-10, ftol=1e-10, gtol=1e-10,
            )
            p = sort_cole_params(res.x)
            cost = float(np.sum(residual_vector(p, freq, z, weight_mode)**2))
            rec = {"start_id": j, "success": bool(res.success), "cost": cost, "nfev": res.nfev, "message": res.message}
            rec.update({name: p[i] for i, name in enumerate(PARAM_NAMES)})
            records.append(rec)
            if res.success and cost < best_cost:
                # savesortafter of parametertoonequantityobject
                res.x = p
                best = res
                best_cost = cost
        except Exception as e:
            records.append({"start_id": j, "success": False, "cost": np.inf, "message": str(e)})
    return best, pd.DataFrame(records)


### Cell 05 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.

- Function descriptions:
  - `summarize_multistart_dispersion(ms_df, top_fraction)`: Summarizes dispersion among the best multi-start solutions as an indicator of sensitivity to local optima.
  - `bootstrap_parameter_uncertainty(freq, z, best_p, n_boot, weight_mode)`: Resamples paired complex residuals, refits bootstrap spectra, and returns parameter uncertainty distributions.
  - `sensitivity_analysis(freq, z, best_p)`: Perturbs each fitted parameter and recomputes residuals to quantify parameter sensitivity.
  - `profile_likelihood_ci(freq, z, best_p, param_index, grid_n, delta_chi2)`: Scans one fixed parameter while re-optimizing the others to approximate a profile-likelihood confidence interval.

In [ ]:
# Cell 5. Parameter: Bootstrap, Multi-Start Dispersion, Profile Likelihood Ci, Sensitivity Analysis
def summarize_multistart_dispersion(ms_df, top_fraction=0.10):
    """if multi-start of quantitylocaloptimal."""
    if ms_df is None or ms_df.empty or "cost" not in ms_df:
        return {}
    ok = ms_df[np.isfinite(ms_df["cost"])].sort_values("cost")
    if ok.empty:
        return {}
    n_top = max(3, int(len(ok) * top_fraction))
    top = ok.head(n_top)
    out = {}
    for name in PARAM_NAMES:
        vals = top[name].to_numpy(dtype=float)
        out[f"{name}_multistart_sd"] = float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else np.nan
        out[f"{name}_multistart_cv_pct"] = float(100*np.nanstd(vals, ddof=1)/(abs(np.nanmedian(vals))+1e-30)) if len(vals)>1 else np.nan
    out["multistart_top_n"] = n_top
    return out
def bootstrap_parameter_uncertainty(freq, z, best_p, n_boot=N_BOOTSTRAP, weight_mode=WEIGHT_MODE_MAIN):
    """
    residual-pair bootstrap:
    1) Generate the fitted curve with the best parameters;
    2) Resample paired complex residuals;
    3) Generate bootstrap spectra and refit them;
    4) The parameter distribution provides the median, SD, 95% CI, and bootstrap RSD.
    """
    if not RUN_BOOTSTRAP:
        return pd.DataFrame(), {}
    freq, z = clean_impedance(freq, z, DROP_IM_POSITIVE)
    z_fit = double_cole_impedance(best_p, freq)
    resid = z - z_fit
    rows = []
    for b in range(n_boot):
        idx = rng.integers(0, len(resid), size=len(resid))
        z_boot = z_fit + resid[idx]
        res_b, _ = fit_double_cole_multistart(freq, z_boot, weight_mode=weight_mode, n_random=max(20, N_RANDOM_STARTS//3), guided_center=best_p)
        if res_b is None or not res_b.success:
            rows.append({"boot_id": b, "success": False})
            continue
        row = {"boot_id": b, "success": True}
        p_b = sort_cole_params(res_b.x)
        row.update({name: p_b[i] for i, name in enumerate(PARAM_NAMES)})
        rows.append(row)
    boot_df = pd.DataFrame(rows)
    summary = {"bootstrap_n": len(boot_df), "bootstrap_success_n": int(boot_df.get("success", pd.Series(dtype=bool)).sum())}
    ok = boot_df[boot_df.get("success", False) == True] if "success" in boot_df else pd.DataFrame()
    for name in PARAM_NAMES:
        if ok.empty or name not in ok:
            summary[f"{name}_boot_median"] = np.nan
            summary[f"{name}_boot_sd"] = np.nan
            summary[f"{name}_boot_ci_low"] = np.nan
            summary[f"{name}_boot_ci_high"] = np.nan
            summary[f"{name}_boot_rsd_pct"] = np.nan
            continue
        vals = ok[name].to_numpy(dtype=float)
        med = np.nanmedian(vals)
        sd = np.nanstd(vals, ddof=1) if len(vals) > 1 else np.nan
        summary[f"{name}_boot_median"] = float(med)
        summary[f"{name}_boot_sd"] = float(sd)
        summary[f"{name}_boot_ci_low"] = float(np.nanpercentile(vals, 2.5))
        summary[f"{name}_boot_ci_high"] = float(np.nanpercentile(vals, 97.5))
        summary[f"{name}_boot_rsd_pct"] = float(100*sd/(abs(med)+1e-30)) if np.isfinite(sd) else np.nan
    return boot_df, summary
def sensitivity_analysis(freq, z, best_p=None):
    """, checkisoneresidual."""
    if not RUN_SENSITIVITY:
        return pd.DataFrame()
    rows = []
    for wm in SENSITIVITY_WEIGHT_MODES:
        res, ms = fit_double_cole_multistart(freq, z, weight_mode=wm, n_random=max(30, N_RANDOM_STARTS//2), guided_center=best_p)
        if res is None or not res.success:
            rows.append({"weight_mode": wm, "success": False})
            continue
        p = sort_cole_params(res.x)
        row = {"weight_mode": wm, "success": True}
        row.update({name: p[i] for i, name in enumerate(PARAM_NAMES)})
        row.update(fit_metrics(*clean_impedance(freq, z, DROP_IM_POSITIVE), p))
        rows.append(row)
    return pd.DataFrame(rows)
def profile_likelihood_ci(freq, z, best_p, param_index, grid_n=PROFILE_GRID_N, delta_chi2=3.84):
    """
    Simplified profile-likelihood CI.Fix one parameter and re-optimize the remaining parameters.
    This step is time-consuming and is disabled by default.Return the acceptable parameter range.
    """
    if not RUN_PROFILE_LIKELIHOOD:
        return {"ci_low": np.nan, "ci_high": np.nan, "grid": pd.DataFrame()}
    freq, z = clean_impedance(freq, z, DROP_IM_POSITIVE)
    lower, upper = dynamic_bounds(freq, z)
    p_best = sort_cole_params(best_p)
    base_cost = np.sum(residual_vector(p_best, freq, z, WEIGHT_MODE_MAIN)**2)
    name = PARAM_NAMES[param_index]
    lo = max(lower[param_index], p_best[param_index] / 10 if p_best[param_index] > 0 else lower[param_index])
    hi = min(upper[param_index], p_best[param_index] * 10 + 1e-30)
    if name.startswith("alpha"):
        lo, hi = max(lower[param_index], p_best[param_index]-0.25), min(upper[param_index], p_best[param_index]+0.25)
    if name.startswith("tau"):
        grid = np.logspace(np.log10(lo), np.log10(hi), grid_n)
    else:
        grid = np.linspace(lo, hi, grid_n)
    records = []
    free_idx = [i for i in range(7) if i != param_index]
    for val in grid:
        p0_free = p_best[free_idx]
        lb_free = lower[free_idx]
        ub_free = upper[free_idx]
        def res_free(x_free):
            p = p_best.copy()
            p[param_index] = val
            p[free_idx] = x_free
            return residual_vector(p, freq, z, WEIGHT_MODE_MAIN)
        try:
            res = least_squares(res_free, p0_free, bounds=(lb_free, ub_free), max_nfev=3000)
            p = p_best.copy(); p[param_index] = val; p[free_idx] = res.x
            cost = np.sum(residual_vector(sort_cole_params(p), freq, z, WEIGHT_MODE_MAIN)**2)
            records.append({"fixed_param": name, "fixed_value": val, "cost": cost, "delta_cost": cost-base_cost, "success": res.success})
        except Exception as e:
            records.append({"fixed_param": name, "fixed_value": val, "cost": np.inf, "delta_cost": np.inf, "success": False, "message": str(e)})
    grid_df = pd.DataFrame(records)
    ok = grid_df[grid_df["delta_cost"] <= delta_chi2]
    if ok.empty:
        return {"ci_low": np.nan, "ci_high": np.nan, "grid": grid_df}
    return {"ci_low": float(ok["fixed_value"].min()), "ci_high": float(ok["fixed_value"].max()), "grid": grid_df}


### Cell 06 - Main Workflow Execution

This cell runs the main workflow or intermediate data-organization steps of the current notebook.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.


In [ ]:
# Cell 6. First-pass batch multi-start fitting
fit_results = {}
all_param_rows = []
all_multistart_rows = []
for sample, sample_data in data_store.items():
    print(f"\n=== Sample: {sample} ===")
    fit_results[sample] = {}
    for dtype in DATA_TYPES:
        fit_results[sample][dtype] = {}
        for proc in PROCESSING_TYPES:
            if proc not in sample_data.get(dtype, {}):
                continue
            freq = sample_data[dtype][proc]["f"]
            z = sample_data[dtype][proc]["z"]
            res, ms_df = fit_double_cole_multistart(freq, z, weight_mode=WEIGHT_MODE_MAIN)
            fit_results[sample][dtype][proc] = {"res": res, "multistart": ms_df}
            if ms_df is not None and not ms_df.empty:
                ms_df = ms_df.copy()
                ms_df.insert(0, "Sample", sample)
                ms_df.insert(1, "Data_Type", dtype)
                ms_df.insert(2, "Processing", proc)
                all_multistart_rows.append(ms_df)
            if res is None or not res.success:
                print(f"  {dtype}-{proc}: failed")
                continue
            p = sort_cole_params(res.x)
            metrics = fit_metrics(*clean_impedance(freq, z, DROP_IM_POSITIVE), p)
            row = {"Sample": sample, "Data_Type": dtype, "Processing": proc, "success": True, "cost": float(np.sum(res.fun**2))}
            row.update({name: p[i] for i, name in enumerate(PARAM_NAMES)})
            row.update(metrics)
            row.update(summarize_multistart_dispersion(ms_df))
            all_param_rows.append(row)
            print(f"  {dtype}-{proc}: cost={row['weighted_mse']:.4g}, Rinf={p[0]:.3g}, tau=({p[2]:.3g},{p[5]:.3g})")
params_initial_df = pd.DataFrame(all_param_rows)
multistart_all_df = pd.concat(all_multistart_rows, ignore_index=True) if all_multistart_rows else pd.DataFrame()
params_initial_df.head()


### Cell 07 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.

- Function descriptions:
  - `flag_param_jump(row, group_median)`: Compares one fitted result with the group median and flags large jumps in resistance, time constant, or exponent.

In [ ]:
# Cell 7. Parametercheckand Group-Guided Refit
def flag_param_jump(row, group_median):
    """checkisrelative type group median."""
    flags = []
    for name in ["R_inf", "R1", "R2"]:
        med = group_median.get(name, np.nan)
        val = row.get(name, np.nan)
        if np.isfinite(med) and med > 0 and np.isfinite(val):
            fold = max(val/(med+1e-30), med/(val+1e-30))
            if fold > RESISTANCE_FOLD_THRESHOLD:
                flags.append(f"{name}_fold>{RESISTANCE_FOLD_THRESHOLD}")
    for name in ["tau1", "tau2"]:
        med = group_median.get(name, np.nan)
        val = row.get(name, np.nan)
        if np.isfinite(med) and med > 0 and np.isfinite(val) and val > 0:
            if abs(np.log10(val) - np.log10(med)) > TAU_LOG10_THRESHOLD:
                flags.append(f"{name}_logjump")
    for name in ["alpha1", "alpha2"]:
        med = group_median.get(name, np.nan)
        val = row.get(name, np.nan)
        if np.isfinite(med) and np.isfinite(val):
            if abs(val - med) > ALPHA_ABS_THRESHOLD:
                flags.append(f"{name}_jump")
    return ";".join(flags)
refit_log = []
if ENABLE_GROUP_GUIDED_REFIT and not params_initial_df.empty:
    grouped_medians = params_initial_df.groupby(["Data_Type", "Processing"])[PARAM_NAMES].median().reset_index()
    median_map = {(r["Data_Type"], r["Processing"]): {p: r[p] for p in PARAM_NAMES} for _, r in grouped_medians.iterrows()}
    for idx, row in params_initial_df.iterrows():
        key = (row["Data_Type"], row["Processing"])
        gm = median_map.get(key, {})
        flag = flag_param_jump(row, gm)
        params_initial_df.loc[idx, "jump_flag"] = flag
        if not flag:
            continue
        sample, dtype, proc = row["Sample"], row["Data_Type"], row["Processing"]
        print(f"Refit triggered: {sample} {dtype}-{proc}: {flag}")
        freq = data_store[sample][dtype][proc]["f"]
        z = data_store[sample][dtype][proc]["z"]
        guided = np.array([gm[p] for p in PARAM_NAMES], dtype=float)
        res_new, ms_new = fit_double_cole_multistart(freq, z, weight_mode=WEIGHT_MODE_MAIN, n_random=N_GROUP_REFIT_STARTS, guided_center=guided)
        if res_new is None or not res_new.success:
            refit_log.append({"Sample": sample, "Data_Type": dtype, "Processing": proc, "accepted": False, "reason": "refit_failed", "old_flag": flag})
            continue
        old_cost = row["weighted_mse"]
        new_p = sort_cole_params(res_new.x)
        new_metrics = fit_metrics(*clean_impedance(freq, z, DROP_IM_POSITIVE), new_p)
        new_cost = new_metrics["weighted_mse"]
        accepted = new_cost <= old_cost * ACCEPT_REFIT_COST_INCREASE
        refit_log.append({"Sample": sample, "Data_Type": dtype, "Processing": proc, "accepted": accepted, "old_cost": old_cost, "new_cost": new_cost, "old_flag": flag})
        if accepted:
            fit_results[sample][dtype][proc] = {"res": res_new, "multistart": ms_new}
            for i, name in enumerate(PARAM_NAMES):
                params_initial_df.loc[idx, name] = new_p[i]
            for k, v in new_metrics.items():
                params_initial_df.loc[idx, k] = v
            params_initial_df.loc[idx, "jump_flag_after_refit"] = flag_param_jump({**row.to_dict(), **{name: new_p[i] for i,name in enumerate(PARAM_NAMES)}}, gm)
refit_log_df = pd.DataFrame(refit_log)
params_after_refit_df = params_initial_df.copy()
refit_log_df.head()


### Cell 08 - Model Fitting and Uncertainty Analysis

This cell performs Cole-Cole/equivalent-circuit parameter fitting, bootstrap resampling, or statistical summarization.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.

In [ ]:
# Cell 8. Bootstrap uncertainty, profile CI, and sensitivity analysis
bootstrap_rows = []
bootstrap_raw_rows = []
sensitivity_rows = []
profile_rows = []
for sample, sample_data in data_store.items():
    print(f"\nUncertainty: {sample}")
    for dtype in DATA_TYPES:
        for proc in PROCESSING_TYPES:
            if proc not in sample_data.get(dtype, {}):
                continue
            obj = fit_results.get(sample, {}).get(dtype, {}).get(proc, {})
            res = obj.get("res")
            if res is None or not res.success:
                continue
            freq = sample_data[dtype][proc]["f"]
            z = sample_data[dtype][proc]["z"]
            p_best = sort_cole_params(res.x)
            boot_df, boot_summary = bootstrap_parameter_uncertainty(freq, z, p_best, n_boot=N_BOOTSTRAP, weight_mode=WEIGHT_MODE_MAIN)
            boot_summary.update({"Sample": sample, "Data_Type": dtype, "Processing": proc})
            bootstrap_rows.append(boot_summary)
            if not boot_df.empty:
                boot_df = boot_df.copy()
                boot_df.insert(0, "Sample", sample)
                boot_df.insert(1, "Data_Type", dtype)
                boot_df.insert(2, "Processing", proc)
                bootstrap_raw_rows.append(boot_df)
            sens_df = sensitivity_analysis(freq, z, best_p=p_best)
            if not sens_df.empty:
                sens_df.insert(0, "Sample", sample)
                sens_df.insert(1, "Data_Type", dtype)
                sens_df.insert(2, "Processing", proc)
                sensitivity_rows.append(sens_df)
            if RUN_PROFILE_LIKELIHOOD:
                for pi, pname in enumerate(PARAM_NAMES):
                    prof = profile_likelihood_ci(freq, z, p_best, pi)
                    profile_rows.append({"Sample": sample, "Data_Type": dtype, "Processing": proc, "Param": pname, "profile_ci_low": prof["ci_low"], "profile_ci_high": prof["ci_high"]})
bootstrap_summary_df = pd.DataFrame(bootstrap_rows)
bootstrap_all_df = pd.concat(bootstrap_raw_rows, ignore_index=True) if bootstrap_raw_rows else pd.DataFrame()
sensitivity_df = pd.concat(sensitivity_rows, ignore_index=True) if sensitivity_rows else pd.DataFrame()
profile_ci_df = pd.DataFrame(profile_rows)
bootstrap_summary_df.head()


### Cell 09 - Read or Write Tabular Data

This cell reads input workbooks or writes computed results to Excel for later plotting, statistics, and checking.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.

In [ ]:
# Cell 9. Saveparametertable, Bootstrap Table And
# Merge the main parameter table with the bootstrap summary.
params_final_df = params_after_refit_df.copy()
if not bootstrap_summary_df.empty:
    params_final_df = params_final_df.merge(bootstrap_summary_df, on=["Sample", "Data_Type", "Processing"], how="left")
# Generate fitted curves and cross-check results（data vs reconstruction of Re/Im）
curve_tables = []
for sample, sample_data in data_store.items():
    for dtype in DATA_TYPES:
        for proc in PROCESSING_TYPES:
            if proc not in sample_data.get(dtype, {}):
                continue
            obj = fit_results.get(sample, {}).get(dtype, {}).get(proc, {})
            res = obj.get("res")
            if res is None or not res.success:
                continue
            freq = sample_data[dtype][proc]["f"]
            z = sample_data[dtype][proc]["z"]
            freq, z = clean_impedance(freq, z, DROP_IM_POSITIVE)
            p = sort_cole_params(res.x)
            z_fit_at_data = double_cole_impedance(p, freq)
            f_smooth = np.logspace(np.log10(np.min(freq)), np.log10(np.max(freq)), 300)
            z_fit_smooth = double_cole_impedance(p, f_smooth)
            mag_floor = REL_SCALE_FLOOR * np.nanmedian(np.abs(z) + 1e-30)
            scale = np.maximum(np.abs(z), mag_floor)
            df_data = pd.DataFrame({
                "Sample": sample, "Data_Type": dtype, "Processing": proc,
                "freq": freq,
                "re_data": z.real, "imag_data": z.imag,
                "re_fit_at_data": z_fit_at_data.real, "imag_fit_at_data": z_fit_at_data.imag,
                "rel_resid_re_pct": 100*(z_fit_at_data.real - z.real)/scale,
                "rel_resid_imag_pct": 100*(z_fit_at_data.imag - z.imag)/scale,
            })
            df_smooth = pd.DataFrame({
                "Sample": sample, "Data_Type": dtype, "Processing": proc,
                "freq_smooth": f_smooth,
                "re_fit_smooth": z_fit_smooth.real,
                "imag_fit_smooth": z_fit_smooth.imag,
            })
            curve_tables.append((sample, dtype, proc, df_data, df_smooth))
out_xlsx = DATA_DIR / "exp_doubleColefit_results.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    params_final_df.to_excel(writer, sheet_name="fit_parameters", index=False)
    if not multistart_all_df.empty:
        multistart_all_df.to_excel(writer, sheet_name="multi_start_all", index=False)
    if not refit_log_df.empty:
        refit_log_df.to_excel(writer, sheet_name="jump_refit_log", index=False)
    if not bootstrap_summary_df.empty:
        bootstrap_summary_df.to_excel(writer, sheet_name="bootstrap_summary", index=False)
    if not bootstrap_all_df.empty:
        bootstrap_all_df.to_excel(writer, sheet_name="bootstrap_samples", index=False)
    if not sensitivity_df.empty:
        sensitivity_df.to_excel(writer, sheet_name="sensitivity", index=False)
    if not profile_ci_df.empty:
        profile_ci_df.to_excel(writer, sheet_name="profile_CI", index=False)
# Save curve tables separately to avoid overly wide Excel sheets.
curve_xlsx = CURVE_DIR / "exp_doubleColefit_curves.xlsx"
with pd.ExcelWriter(curve_xlsx, engine="openpyxl") as writer:
    for sample, dtype, proc, df_data, df_smooth in curve_tables:
        sname = re.sub(r"[\\/*?:\[\]]", "_", f"{sample}_{dtype}_{proc}")[:31]
        df_data.to_excel(writer, sheet_name=sname, index=False)
    # tablesave as CSV, to facilitate Origin or Python later
    smooth_all = pd.concat([x[4] for x in curve_tables], ignore_index=True) if curve_tables else pd.DataFrame()
    if not smooth_all.empty:
        smooth_all.to_excel(writer, sheet_name="smooth_fit_long", index=False)
print("Saved:", out_xlsx)
print("Saved:", curve_xlsx)


### Cell 10 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.

- Function descriptions:
  - `plot_fit_assessment(sample, out_dir)`: Plots Nyquist, Bode, and residual diagnostics for experimental spectra and double Cole-Cole fits.

In [ ]:
# Cell 10. Plottingsave: Nyquist + Re/Im Cross + Forresidual
def plot_fit_assessment(sample, out_dir=FIG_DIR):
    sample_data = data_store[sample]
    nrows = len(DATA_TYPES)
    fig, axes = plt.subplots(nrows, 4, figsize=(22, 4.8*nrows), constrained_layout=True)
    if nrows == 1:
        axes = np.array([axes])
    for r, dtype in enumerate(DATA_TYPES):
        ax_nyq, ax_re, ax_im, ax_res = axes[r]
        for proc, color, marker in [("Exp", "tab:red", "o"), ("GP", "tab:blue", "x")]:
            if proc not in sample_data.get(dtype, {}):
                continue
            obj = fit_results.get(sample, {}).get(dtype, {}).get(proc, {})
            res = obj.get("res")
            if res is None or not res.success:
                continue
            freq = sample_data[dtype][proc]["f"]
            z = sample_data[dtype][proc]["z"]
            freq, z = clean_impedance(freq, z, DROP_IM_POSITIVE)
            p = sort_cole_params(res.x)
            z_fit = double_cole_impedance(p, freq)
            f_smooth = np.logspace(np.log10(freq.min()), np.log10(freq.max()), 300)
            z_smooth = double_cole_impedance(p, f_smooth)
            mag_floor = REL_SCALE_FLOOR * np.nanmedian(np.abs(z)+1e-30)
            scale = np.maximum(np.abs(z), mag_floor)
            rel_re = 100*(z_fit.real-z.real)/scale
            rel_im = 100*(z_fit.imag-z.imag)/scale
            ax_nyq.scatter(z.real, -z.imag, s=18, color=color, marker=marker, alpha=0.55, label=f"{proc} data")
            ax_nyq.plot(z_smooth.real, -z_smooth.imag, color=color, lw=2.0, label=f"{proc} fit")
            ax_re.scatter(freq, z.real, s=16, color=color, marker=marker, alpha=0.45)
            ax_re.plot(f_smooth, z_smooth.real, color=color, lw=2.0, label=f"{proc}")
            ax_im.scatter(freq, -z.imag, s=16, color=color, marker=marker, alpha=0.45)
            ax_im.plot(f_smooth, -z_smooth.imag, color=color, lw=2.0, label=f"{proc}")
            ax_res.scatter(freq, rel_re, s=14, color=color, marker=marker, alpha=0.55, label=f"{proc} Re")
            ax_res.scatter(freq, rel_im, s=14, facecolors='none', edgecolors=color, marker=marker, alpha=0.55, label=f"{proc} Im")
        ax_nyq.set_aspect("equal", adjustable="box")
        ax_nyq.set_xlabel("Z' / Ohm")
        ax_nyq.set_ylabel(f"{dtype}\n-Z'' / Ohm")
        ax_nyq.grid(alpha=0.25)
        ax_nyq.legend(fontsize=8)
        ax_re.set_xscale("log"); ax_re.set_xlabel("f / Hz"); ax_re.set_ylabel("Z' / Ohm"); ax_re.grid(alpha=0.25); ax_re.legend(fontsize=8)
        ax_im.set_xscale("log"); ax_im.set_xlabel("f / Hz"); ax_im.set_ylabel("-Z'' / Ohm"); ax_im.grid(alpha=0.25); ax_im.legend(fontsize=8)
        ax_res.axhline(0, color="black", lw=1, ls="--")
        ax_res.set_xscale("log"); ax_res.set_xlabel("f / Hz"); ax_res.set_ylabel("Relative residual / %")
        ax_res.grid(alpha=0.25); ax_res.legend(fontsize=7, ncol=2)
    fig.suptitle(f"Double Cole fitting assessment: {sample}", fontsize=16)
    out_path = out_dir / f"{re.sub(r'[^0-9A-Za-z_\-]+','_',str(sample))}_doubleCole_fit.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return out_path
for sample in data_store.keys():
    p = plot_fit_assessment(sample)
    print("Saved", p)


### Cell 11 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.

- Function descriptions:
  - `benjamini_hochberg(pvals)`: Takes a set of p values and returns Benjamini-Hochberg FDR-adjusted q values.
  - `rank_biserial_from_pairs(x, y)`: Computes the rank-biserial effect size for a paired Wilcoxon comparison.

In [ ]:
# Cell 11. statistical analysis：paired Wilcoxon + FDR + paired effect size
def benjamini_hochberg(pvals):
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan)
    mask = np.isfinite(pvals)
    p = pvals[mask]
    m = len(p)
    if m == 0:
        return qvals
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * m / (np.arange(1, m+1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    q_unsorted = np.empty_like(q)
    q_unsorted[order] = q
    qvals[mask] = q_unsorted
    return qvals
def rank_biserial_from_pairs(x, y):
    d = np.asarray(y) - np.asarray(x)
    d = d[np.isfinite(d) & (d != 0)]
    if len(d) == 0:
        return np.nan
    ranks = stats.rankdata(np.abs(d))
    Wpos = np.sum(ranks[d > 0])
    Wneg = np.sum(ranks[d < 0])
    return float((Wpos - Wneg) / (Wpos + Wneg + 1e-30))
# Build metrics for statistical testing: fitting errors plus bootstrap RSD/CI widths.
stat_rows_long = []
metric_cols = ["weighted_mse", "rel_complex_rmse", "nrmse_re", "nrmse_im"]
for pname in PARAM_NAMES:
    if f"{pname}_boot_rsd_pct" in params_final_df.columns:
        metric_cols.append(f"{pname}_boot_rsd_pct")
    if f"{pname}_boot_ci_low" in params_final_df.columns and f"{pname}_boot_ci_high" in params_final_df.columns:
        params_final_df[f"{pname}_boot_ci_width"] = params_final_df[f"{pname}_boot_ci_high"] - params_final_df[f"{pname}_boot_ci_low"]
        metric_cols.append(f"{pname}_boot_ci_width")
stat_results = []
for dtype in DATA_TYPES:
    sub = params_final_df[params_final_df["Data_Type"] == dtype]
    for metric in metric_cols:
        if metric not in sub.columns:
            continue
        piv = sub.pivot_table(index="Sample", columns="Processing", values=metric, aggfunc="median")
        if not {"Exp", "GP"}.issubset(piv.columns):
            continue
        paired = piv[["Exp", "GP"]].dropna()
        if len(paired) < 3:
            continue
        try:
            stat, pval = stats.wilcoxon(paired["Exp"], paired["GP"], zero_method="wilcox", alternative="two-sided")
        except Exception:
            stat, pval = np.nan, np.nan
        diff = paired["GP"] - paired["Exp"]
        stat_results.append({
            "Data_Type": dtype,
            "Metric": metric,
            "n_pairs": len(paired),
            "median_Exp": float(np.nanmedian(paired["Exp"])),
            "median_GP": float(np.nanmedian(paired["GP"])),
            "median_paired_diff_GP_minus_Exp": float(np.nanmedian(diff)),
            "HL_approx_median_diff": float(np.nanmedian(diff)),
            "wilcoxon_stat": float(stat) if np.isfinite(stat) else np.nan,
            "p_value": float(pval) if np.isfinite(pval) else np.nan,
            "rank_biserial_effect": rank_biserial_from_pairs(paired["Exp"], paired["GP"]),
        })
stats_df = pd.DataFrame(stat_results)
if not stats_df.empty:
    stats_df["q_value_BH_FDR"] = benjamini_hochberg(stats_df["p_value"].to_numpy())
stats_path = DATA_DIR / "exp_doubleColefit_paired_statistics.xlsx"
with pd.ExcelWriter(stats_path, engine="openpyxl") as writer:
    stats_df.to_excel(writer, sheet_name="wilcoxon_FDR", index=False)
    params_final_df.to_excel(writer, sheet_name="fit_parameters", index=False)
print("Saved statistics:", stats_path)
stats_df.head(20)


### Cell 12 - Main Workflow Execution

This cell runs the main workflow or intermediate data-organization steps of the current notebook.

- Purpose: Double Cole model fitting, bootstrap statistics, and result summarization for experimental impedance spectra.
